# 2. One GENIE3 network per subtype

Every network is inferred at the same sample size. Without that, Luminal A looks more
structured than Basal simply because it has three times as many tumours.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
from src.network import annotate_edges, link_list, matched_networks
from src.preprocessing import split_by_group
from src.signatures import load_tfs

expr = read_matrix(results_path("preprocessed", "tcga_expression.tsv.gz"))
meta = read_table(results_path("preprocessed", "tcga_meta.tsv"), index_col=0)
groups = split_by_group(expr, meta["subtype"], cfg.subtypes, cfg.min_subtype_n)
{name: sub.shape[1] for name, sub in groups.items()}

In [ ]:
n_match = cfg.n_match or min(sub.shape[1] for sub in groups.values())
n_match

Her2 has 67 TCGA tumours and would drag `n_match` down for everyone, which is why
`min_subtype_n` is 100 by default. `config_her2.json` keeps it, at that cost.

In [ ]:
# minutes at these settings; drop n_trees to 50 while exploring
networks = matched_networks(
    groups, n_match=n_match, n_reps=cfg.n_match_reps, seed=cfg.seed, verbose=True,
    tree_method=cfg.tree_method, n_trees=cfg.n_trees,
    max_features=cfg.max_features, n_jobs=cfg.n_jobs,
)

In [ ]:
tfs = load_tfs()
for name, vim in networks.items():
    edges = annotate_edges(link_list(vim, top_n=cfg.top_edges), tfs, name)
    print(name, "|", edges.iloc[0]["regulator"], "->", edges.iloc[0]["target"],
          "|", f"{edges['regulator_is_tf'].mean():.0%} TF regulators")

## Only TFs as regulators

Allowing all 500 genes to regulate gives predictive associations. Restricting to the
Lambert TF list gives edges that could plausibly be regulation. Both are worth reporting;
`scripts/02_subtype_networks.py --tf-only` writes the restricted version alongside.

In [ ]:
regulators = [g for g in expr.index if g in set(tfs)]
print(len(regulators), "of", expr.shape[0], "selected genes are transcription factors")